# Qwen3.8 Flash-Next NVFP4 — Pennyroyal FR-Spec + Cloudflare

Notebook autônomo para preparar um **Colab vazio com GPU compatível** e publicar uma API temporária. Execute **Ambiente de execução → Executar tudo**. Selecione o ambiente **G4, com 48 processadores, Blackwell SM120 de 96 GB e RAM alta**; o código verifica o hardware real, independentemente do nome da opção no Colab.

**Receita:** `pennyroyal-v2.5.0`, commit `2c675da096939cb01102f8f4871bda3db55f7f28`. **Modelo:** `RadixArk/Qwen3.8-Flash-Next-NVFP4`, revisão `7b719225242aacd3dbd3f9407468c2ee9a9d2594`.

- Launcher oficial `serve-flash-next-frspec.sh`, sem alterar seus parâmetros de inferência: contexto 524.288, NEXTN MTP nativo + FR-Spec 64K, KV FP8 e HiCache/NIXL.
- Instalação nativa, sem Docker e sem venv. Um Python 3.12.13 separado, gerenciado pelo `uv`, segue a versão da receita e mantém o kernel do Colab intacto.
- Downloads paralelos com Hugging Face/Xet e seleção de pesos pelo índice. O espaço necessário é calculado; não se usa o tamanho total do repositório como requisito.
- Chave gerada por `secrets.token_urlsafe(32)` em cada inicialização do servidor. Nenhuma chave vem gravada neste arquivo.
- Cloudflare Quick Tunnel sem conta, domínio ou token. **Use `stream: false`: Quick Tunnels não suportam oficialmente SSE.** Requisições longas também estão sujeitas aos timeouts da Cloudflare.

**Limite de validação:** estrutura, sintaxe e lógica auxiliar verificadas; este notebook não foi executado em uma GPU Colab nesta entrega. O autor da receita qualifica Fedora; a instalação Ubuntu/Colab abaixo é uma adaptação. Só a última célula declara sucesso após verificar autenticação e uma geração pela URL pública.

Preparado em 13/09/2026. O tempo inicial inclui downloads, compilação de kernels, hash dos pesos e captura de CUDA graphs. Os logs permanecem visíveis durante a espera. O armazenamento e a URL duram apenas enquanto a instância e os processos existirem.

## 0. Token Hugging Face — opcional

Cole seu **User Access Token** com permissão de leitura no campo oculto ao executar a célula. Pressione **Enter sem preencher** para continuar sem token. Crie seu token em [Hugging Face → Access Tokens](https://huggingface.co/settings/tokens).

O token fica no ambiente da sessão, sem ser gravado no código nem exibido na saída. Os downloads seguintes usam essa autenticação automaticamente. Isso pode ajudar com limites de acesso, mas não garante maior velocidade. Os 32 downloads simultâneos e o modo Xet de alto desempenho continuam ativos. Este token é diferente da chave temporária da API de inferência.

In [ ]:
import os
from getpass import getpass

_hf_token = getpass('Token Hugging Face (opcional; Enter para continuar sem token): ').strip()
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    print('Token Hugging Face configurado para os downloads desta sessão.')
else:
    os.environ.pop('HF_TOKEN', None)
    print('Sem token Hugging Face informado; continuando com acesso público.')
# Atualiza também ENV se esta célula for repetida depois da configuração.
if isinstance(globals().get('ENV'), dict):
    if _hf_token:
        ENV['HF_TOKEN'] = _hf_token
    else:
        ENV.pop('HF_TOKEN', None)
del _hf_token

## 1. Configuração e verificação antes dos downloads

O orçamento de RAM é conservador: 47,68 GiB para PLE + 32 GiB para HiCache + 16 GiB de margem = 95,68 GiB disponíveis. Não é um mínimo medido pelo autor. Mantemos o backend PLE padrão em RAM, sem criar a cópia adicional de aproximadamente 48 GiB do modo NVMe.

O driver pertence à infraestrutura Colab. Com R580/R590/R595, este notebook instala `cuda-compat-13-3` e prioriza suas bibliotecas de usuário em `LD_LIBRARY_PATH`. Com R610+, usa o driver nativo. Não substitui o módulo NVIDIA do kernel. Antes de instalar SGLang ou baixar pesos, um processo separado verifica `cuInit`, a biblioteca efetivamente carregada e PTX JIT CUDA 13.3. A aceitação dessa combinação pela GPU/VM é verificada em execução.

In [ ]:
import os, sys, json, subprocess, shutil, time, secrets, re, signal, shlex, platform
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

ROOT = Path('/content/pennyroyal-colab')
REPO = ROOT / 'source'
MODEL = ROOT / 'model'
LOGS = ROOT / 'logs'
RUNTIME_TAG = 'pennyroyal-v2.5.0'
RUNTIME_COMMIT = '2c675da096939cb01102f8f4871bda3db55f7f28'
MODEL_ID = 'RadixArk/Qwen3.8-Flash-Next-NVFP4'
MODEL_REV = '7b719225242aacd3dbd3f9407468c2ee9a9d2594'
NIXL_REV = 'aecbc3846d92c34c7507a58d776e1fda50ff4fba'
STARTUP_TIMEOUT = 7200  # segundos; inclui hash, compilação e CUDA graphs
DOWNLOAD_WORKERS = 32  # arquivos simultâneos; Xet em alto desempenho
BUILD_JOBS = 48  # G4 dedicado: até 48 jobs de compilação
NVCC_THREADS = 4  # paralelismo interno de cada compilação FlashInfer/NVCC
WEIGHT_LOAD_WORKERS = 24  # separado dos jobs: shards grandes pressionam a RAM
ONLINE_FP8 = False  # opcional oficial; False preserva a receita padrão
# Uma nova execução completa encerra os processos anteriores deste notebook.
if callable(globals().get('stop_process')):
    for old_process in ('tunnel_process', 'server_process'):
        stop_process(globals().get(old_process))
for directory in (ROOT, LOGS):
    directory.mkdir(parents=True, exist_ok=True)

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Selecione GPU Blackwell SM120 de 96 GB no Colab antes de executar.')
query = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version',
    '--format=csv,noheader,nounits'], text=True).strip()
print(query)
gpu_name, memory_mib, capability, driver = [x.strip() for x in query.splitlines()[0].split(',')]
if capability != '12.0' or float(memory_mib) < 90000:
    raise RuntimeError(f'GPU incompatível com esta receita: {gpu_name}, SM {capability}, {memory_mib} MiB. Necessária SM120 de 96 GB.')
DRIVER_BRANCH = int(driver.split('.')[0])
if DRIVER_BRANCH < 580:
    raise RuntimeError(f'Driver {driver} fora das versões contempladas por este notebook (R580+).')
USE_CUDA_COMPAT = DRIVER_BRANCH < 610
if USE_CUDA_COMPAT and DRIVER_BRANCH not in (580, 590, 595):
    raise RuntimeError(f'Ramo {DRIVER_BRANCH} não listado na matriz CUDA compat 13.3 usada pelo notebook.')
print(f'Driver do host: {driver}; CUDA 13.3 via ' +
      ('cuda-compat-13-3 (bibliotecas de usuário)' if USE_CUDA_COMPAT else 'driver nativo'))
if platform.machine() != 'x86_64':
    raise RuntimeError('A instalação abaixo foi preparada para Linux x86_64.')
os_release = dict(line.split('=', 1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
OS_VERSION = os_release.get('VERSION_ID', '').strip('"')
if os_release.get('ID', '').strip('"') != 'ubuntu' or OS_VERSION not in ('22.04', '24.04'):
    raise RuntimeError(f'Ubuntu não contemplado pelo instalador: {os_release}. Esperado 22.04 ou 24.04.')
meminfo = {line.split(':')[0]: int(line.split()[1]) * 1024 for line in Path('/proc/meminfo').read_text().splitlines()}
ram_gib = meminfo['MemAvailable'] / 2**30
required_ram = 47.68 + 32 + 16
print(f'RAM disponível: {ram_gib:.2f} GiB; orçamento conservador: {required_ram:.2f} GiB.')
if ram_gib < required_ram:
    raise RuntimeError('RAM insuficiente para o orçamento desta configuração. Selecione RAM alta; não reduzimos HiCache nem trocamos o backend silenciosamente.')
print(f'Disco disponível: {shutil.disk_usage(ROOT).free / 2**30:.2f} GiB')
print('Verificação inicial OK —', datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%d/%m/%Y %H:%M:%S'))

ENV = os.environ.copy()
ENV.update({
    'DEBIAN_FRONTEND': 'noninteractive', 'PYTHONUNBUFFERED': '1',
    'CUDA_VISIBLE_DEVICES': '0', 'CUDA_HOME': '/usr/local/cuda-13.3',
    'CUDACXX': '/usr/local/cuda-13.3/bin/nvcc',
    'CC': '/usr/bin/gcc-15', 'CXX': '/usr/bin/g++-15', 'CUDAHOSTCXX': '/usr/bin/g++-15',
    'TORCH_CUDA_ARCH_LIST': '12.0', 'PENNY_BUILD_JOBS': str(BUILD_JOBS),
    'MAX_JOBS': str(BUILD_JOBS), 'CMAKE_BUILD_PARALLEL_LEVEL': str(BUILD_JOBS),
    'CARGO_BUILD_JOBS': str(BUILD_JOBS), 'MAKEFLAGS': f'-j{BUILD_JOBS}', 'FLASHINFER_NINJA_JOBS': str(BUILD_JOBS),
    'FLASHINFER_NVCC_THREADS': str(NVCC_THREADS), 'TORCHINDUCTOR_COMPILE_THREADS': str(BUILD_JOBS),
    'UV_PYTHON_INSTALL_DIR': str(ROOT / 'python'), 'UV_CACHE_DIR': str(ROOT / 'uv-cache'),
    'HF_HOME': str(ROOT / 'cache' / 'huggingface'), 'HF_XET_HIGH_PERFORMANCE': '1',
    'HF_XET_CHUNK_CACHE_SIZE_BYTES': '0', 'HF_HUB_DOWNLOAD_TIMEOUT': '120',
    'HF_HUB_ETAG_TIMEOUT': '60', 'NIXL_PREFIX': str(ROOT / 'nixl-install'),
})
ENV['PATH'] = '/usr/local/cuda-13.3/bin:' + str(Path.home() / '.cargo/bin') + ':' + ENV['PATH']
# Compat antes de qualquer libcuda do host; nunca incluir diretórios CUDA stubs.
# Aplicado apenas aos subprocessos do notebook, sem alterar o driver do kernel.
ENV['PENNY_USE_CUDA_COMPAT'] = '1' if USE_CUDA_COMPAT else '0'
compat_prefix = '/usr/local/cuda-13.3/compat:' if USE_CUDA_COMPAT else ''
ENV['LD_LIBRARY_PATH'] = compat_prefix + str(ROOT / 'nixl-install/lib64') + ':/usr/local/cuda-13.3/lib64:' + ENV.get('LD_LIBRARY_PATH', '')

# Executa em subprocesso com log incremental, heartbeat e erro propagado.
def run(command, name, *, cwd=None, timeout=7200):
    log = LOGS / (name + '.log')
    with log.open('w') as out:
        proc = subprocess.Popen([str(x) for x in command], cwd=cwd, env=ENV,
                                stdout=out, stderr=subprocess.STDOUT, start_new_session=True)
    start = time.monotonic()
    last_heartbeat = start
    try:
        with log.open() as reader:
            while True:
                chunk = reader.read()
                if chunk:
                    print(chunk, end='', flush=True)
                if proc.poll() is not None:
                    print(reader.read(), end='', flush=True)
                    break
                now = time.monotonic()
                if now - start > timeout:
                    raise TimeoutError(f'{name} excedeu {timeout}s; log: {log}')
                if now - last_heartbeat >= 30:
                    print(f'[{name}] em execução há {(now-start)/60:.1f} min; log: {log}', flush=True)
                    last_heartbeat = now
                time.sleep(1)
        if proc.returncode:
            raise RuntimeError(f'{name} terminou com código {proc.returncode}. Consulte o erro acima e {log}.')
    except BaseException:
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()
        raise

def bash(script, name, *, cwd=None, timeout=7200):
    path = ROOT / (name + '.sh')
    path.write_text('set -euo pipefail\n' + script)
    run(['bash', path], name, cwd=cwd, timeout=timeout)

## 2. Instalar toolkit, compiladores, SGLang e NIXL POSIX

Antes do primeiro APT, instala o keyring NVIDIA via download HTTPS e `dpkg`, consolida entradas CUDA duplicadas em `.list` e `.sources` e mantém uma única fonte com `Signed-By` explícito. Preserva outros repositórios e guarda backup das alterações. A normalização também funciona ao repetir a célula. Instala somente componentes de espaço de usuário do CUDA 13.3. GCC 15 vem do repositório Ubuntu Toolchain quando indisponível no repositório padrão. O código usa a instalação nativa documentada pelo projeto e permite até 48 jobs de compilação (Ninja/CMake/Cargo/Make), com quatro threads internos por processo NVCC no FlashInfer. O loader usa 24 workers. Os limites são independentes: podem existir mais threads executáveis que CPUs, e o pico de RAM pode aumentar. Esses valores não garantem ocupação integral da CPU em etapas sequenciais ou de medição na GPU.

Os pesos são baixados depois que as dependências e a execução CUDA forem verificadas. Assim uma falha de compilação ou incompatibilidade do driver aparece antes do download grande.

In [ ]:
# Bootstrap pequeno no Python do Colab; servidor usa o Python separado abaixo.
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'], 'bootstrap-uv')
UV = shutil.which('uv')
if not UV:
    raise RuntimeError('O executável uv não apareceu no PATH após a instalação.')

# Bootstrap do repositório SEM APT: o Colab pode começar com Signed-By conflitante.
import urllib.request
from urllib.parse import urlsplit

def normalize_cuda_sources(apt_root, backup_root, cuda_uri):
    apt_root, backup_root = Path(apt_root), Path(backup_root)
    canonical = apt_root / 'sources.list.d/pennyroyal-cuda.list'
    keyring = '/usr/share/keyrings/cuda-archive-keyring.gpg'
    target = urlsplit(cuda_uri)
    def matches(uri):
        parsed = urlsplit(uri)
        return (parsed.hostname == target.hostname and
                parsed.path.rstrip('/') == target.path.rstrip('/'))
    def save(path, before, after):
        if before == after:
            return
        destination = backup_root / path.relative_to(apt_root)
        destination.parent.mkdir(parents=True, exist_ok=True)
        if path.exists() and not destination.exists():
            shutil.copy2(path, destination)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(after)
        print('Fonte APT normalizada:', path)
    paths = [apt_root / 'sources.list']
    parts = apt_root / 'sources.list.d'
    paths += sorted(parts.glob('*.list')) + sorted(parts.glob('*.sources'))
    for path in paths:
        if not path.is_file() or path == canonical:
            continue
        before = path.read_text()
        if path.suffix != '.sources':
            lines = []
            for line in before.splitlines(keepends=True):
                match = re.match(r'^\s*deb(?:-src)?\s+(?:\[[^\]]*\]\s+)?(\S+)', line)
                if match and matches(match.group(1)):
                    line = '# pennyroyal: CUDA consolidado — ' + line
                lines.append(line)
            after = ''.join(lines)
        else:
            # Deb822: remove somente a URI CUDA, preservando outras URIs/campos.
            blocks = re.split(r'(\n[ \t]*\n)', before)
            for i in range(0, len(blocks), 2):
                block = blocks[i]
                field = re.search(r'^URIs:[^\n]*(?:\n[ \t]+[^\n]*)*', block, re.M | re.I)
                if not field:
                    continue
                uris = field.group(0).split(':', 1)[1].split()
                remaining = [uri for uri in uris if not matches(uri)]
                if len(remaining) == len(uris):
                    continue
                if remaining:
                    blocks[i] = block[:field.start()] + 'URIs: ' + ' '.join(remaining) + block[field.end():]
                else:
                    blocks[i] = '\n'.join('# pennyroyal: CUDA consolidado — ' + line for line in block.split('\n'))
            after = ''.join(blocks)
        save(path, before, after)
    expected = f'deb [signed-by={keyring}] {cuda_uri} /\n'
    save(canonical, canonical.read_text() if canonical.exists() else '', expected)

cuda_uri = f'https://developer.download.nvidia.com/compute/cuda/repos/ubuntu{OS_VERSION.replace(".", "")}/x86_64/'
keyring_deb = ROOT / 'cuda-keyring.deb'
last_error = None
for attempt in range(5):
    try:
        with urllib.request.urlopen(cuda_uri + 'cuda-keyring_1.1-1_all.deb', timeout=60) as response:
            keyring_deb.write_bytes(response.read())
        last_error = None
        break
    except OSError as error:
        last_error = error
        time.sleep(2)
if last_error is not None:
    raise RuntimeError('Não foi possível baixar o keyring oficial NVIDIA.') from last_error
run(['dpkg', '-i', keyring_deb], 'cuda-keyring')
if not Path('/usr/share/keyrings/cuda-archive-keyring.gpg').is_file():
    raise RuntimeError('O keyring NVIDIA não foi instalado.')
backup_root = ROOT / 'apt-source-backups' / datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%Y%m%d-%H%M%S-%f')
normalize_cuda_sources('/etc/apt', backup_root, cuda_uri)
print('Repositório CUDA único com Signed-By explícito. Backup:', backup_root)

bash(r'''
apt-get update -qq
apt-get install -y --no-install-recommends ca-certificates curl wget git build-essential \
  software-properties-common pkg-config libaio-dev liburing-dev libnuma-dev \
  libssl-dev libffi-dev zlib1g-dev cmake ninja-build ffmpeg
if ! apt-cache show gcc-15 >/dev/null 2>&1; then
  add-apt-repository -y ppa:ubuntu-toolchain-r/test
  apt-get update -qq
fi
apt-get install -y --no-install-recommends gcc-15 g++-15
apt-get update -qq
apt-get install -y --no-install-recommends cuda-compiler-13-3 cuda-libraries-dev-13-3 \
  cuda-nvtx-13-3 cuda-cccl-13-3
if [ "$PENNY_USE_CUDA_COMPAT" = "1" ]; then
  apt-get install -y --no-install-recommends cuda-compat-13-3
  test -r /usr/local/cuda-13.3/compat/libcuda.so.1
fi
if ! command -v cargo >/dev/null 2>&1; then
  curl --fail --location --retry 5 https://sh.rustup.rs -o /tmp/penny-rustup.sh
  sh /tmp/penny-rustup.sh -y --profile minimal
fi
apt-get clean
nvcc --version
gcc-15 --version
cargo --version
''', 'toolchain')

# Força PTX JIT de CUDA 13.3 para detectar também a capacidade do driver.
ptx_source = ROOT / 'driver_probe.cu'
ptx_source.write_text('extern "C" __global__ void penny_probe() {}\n')
run([ENV['CUDACXX'], '-ptx', '-arch=compute_120', ptx_source, '-o', ROOT / 'driver_probe.ptx'], 'compile-ptx')
probe = ROOT / 'probe_driver.py'
probe.write_text(r'''
import ctypes, sys
cu = ctypes.CDLL('libcuda.so.1')
rc = cu.cuInit(0)
errors = {803: 'Incompatibilidade entre bibliotecas CUDA e driver do kernel',
          804: 'Forward compatibility não suportada nesta GPU/VM'}
if rc:
    raise RuntimeError(f'cuInit falhou: {rc}: {errors.get(rc, "erro CUDA")}. O servidor não será iniciado.')
version = ctypes.c_int()
assert cu.cuDriverGetVersion(ctypes.byref(version)) == 0
print('CUDA Driver API efetiva:', version.value, flush=True)
from pathlib import Path
maps = Path('/proc/self/maps').read_text()
loaded = sorted({line.split()[-1] for line in maps.splitlines() if '/libcuda.so' in line})
print('libcuda carregada:', loaded, flush=True)
import os
if os.environ.get('PENNY_USE_CUDA_COMPAT') == '1':
    assert any('/cuda-13.3/compat/' in x for x in loaded), 'libcuda de compatibilidade não foi selecionada'
ctx = ctypes.c_void_p()
assert cu.cuDevicePrimaryCtxRetain(ctypes.byref(ctx), 0) == 0
assert cu.cuCtxSetCurrent(ctx) == 0
module = ctypes.c_void_p()
ptx = open(sys.argv[1], 'rb').read() + b'\0'
rc = cu.cuModuleLoadData(ctypes.byref(module), ctypes.c_char_p(ptx))
assert rc == 0, f'Driver não carregou PTX CUDA 13.3: erro CUDA {rc}'
assert cu.cuModuleUnload(module) == 0
print('Driver/PTX JIT CUDA 13.3 OK')
''')
run([sys.executable, probe, ROOT / 'driver_probe.ptx'], 'verify-driver-jit')

run([UV, 'python', 'install', '3.12.13'], 'python-install')
PYTHON = subprocess.check_output([UV, 'python', 'find', '--managed-python', '3.12.13'], env=ENV, text=True).strip()
PYBIN = str(Path(PYTHON).parent)
ENV['PATH'] = PYBIN + ':' + ENV['PATH']
# Não cria venv e não modifica o torch já carregado no kernel do Colab.
def pip_install(args, name, *, cwd=None):
    run([UV, 'pip', 'install', '--python', PYTHON, '--system', '--break-system-packages',
         '--no-cache', *args], name, cwd=cwd)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', RUNTIME_TAG, '--single-branch',
         'https://github.com/jpezzulli/sglang-rtxpro6000.git', REPO], 'clone-sglang')
actual_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != RUNTIME_COMMIT:
    raise RuntimeError(f'Fonte inesperada: {actual_commit}; esperado {RUNTIME_COMMIT}.')
pip_install(['pip', 'setuptools>=80.9.0', 'setuptools-rust>=1.10', 'setuptools-scm>=8.0',
             'wheel', 'build', 'meson', 'meson-python', 'ninja', 'cmake', 'tomlkit', 'pybind11', 'patchelf', 'pyyaml', 'types-PyYAML', 'pytest'], 'python-build-tools')
# Só o checkout SGLang usa os build tools instalados acima.
# Dependências (incluindo cuda-tile/wheel_stub) mantêm isolamento PEP 517.
pip_install(['--prerelease=allow', '--index-strategy', 'unsafe-best-match',
             '--extra-index-url', 'https://docs.sglang.ai/whl/cu130/',
             '--no-build-isolation-package', 'sglang', '-e', 'python'], 'install-sglang', cwd=REPO)

nixl_source = ROOT / 'nixl-source'
if not (nixl_source / '.git').exists():
    run(['git', 'clone', 'https://github.com/ai-dynamo/nixl.git', nixl_source], 'clone-nixl')
run(['git', 'checkout', '--detach', NIXL_REV], 'pin-nixl', cwd=nixl_source)
# Compila contra o torch do servidor; evita o torch 2.11 do build isolado upstream.
run([PYTHON, 'contrib/tomlutil.py', '--wheel-name', 'nixl-cu13', 'pyproject.toml'], 'name-nixl-cu13', cwd=nixl_source)
pip_install(['--no-build-isolation', '--no-deps',
             '--config-settings=setup-args=-Denable_plugins=POSIX',
             '--config-settings=setup-args=-Dnixl_cuda_arch_list=120',
             '--config-settings=setup-args=-Dbuild_tests=false',
             '--config-settings=setup-args=-Dbuild_examples=false', '.'], 'install-nixl-python', cwd=nixl_source)
bash(r'''
python contrib/tomlutil.py --wheel-name nixl-cu13 pyproject.toml
if [ ! -f build-posix/build.ninja ]; then
  meson setup build-posix --buildtype=release \
    --prefix="$NIXL_PREFIX" --libdir=lib64 \
    -Denable_plugins=POSIX -Dnixl_cuda_arch_list=120 \
    -Dbuild_tests=false -Dbuild_examples=false
fi
ninja -C build-posix -j "$PENNY_BUILD_JOBS" install
''', 'build-nixl-posix', cwd=nixl_source)
meta_wheels = sorted((nixl_source / 'build-posix/src/bindings/python/nixl-meta').glob('nixl-*-py3-none-any.whl'))
if len(meta_wheels) != 1:
    raise RuntimeError(f'Esperado um wheel meta NIXL; encontrados: {meta_wheels}')
pip_install([str(meta_wheels[0])], 'install-nixl-meta')
pip_install(['huggingface_hub[hf_xet]', 'requests'], 'download-tools')

verify = ROOT / 'verify_runtime.py'
verify.write_text(r'''
import importlib.metadata as m
import torch, flashinfer, sglang
from nixl._api import nixl_agent, nixl_agent_config
for name in ('sglang', 'torch', 'flashinfer-python', 'nixl-cu13'):
    print(name, m.version(name), flush=True)
assert torch.cuda.is_available(), 'CUDA indisponível no Python do servidor'
assert torch.version.cuda and torch.version.cuda.startswith('13.'), f'Torch CUDA inesperado: {torch.version.cuda}'
assert torch.cuda.get_device_capability(0) == (12, 0)
x = torch.randn((512, 512), device='cuda', dtype=torch.bfloat16)
y = x @ x
assert torch.isfinite(y).all().item()
torch.cuda.synchronize()
print('CUDA matmul OK:', torch.cuda.get_device_name(0), flush=True)
a = nixl_agent('penny-colab-preflight', nixl_agent_config(backends=[]))
plugins = a.get_plugin_list()
print('NIXL plugins:', plugins, flush=True)
assert 'POSIX' in plugins, 'Plugin POSIX não carregado'
a.create_backend('POSIX', {'use_uring': 'true'})
# O_DIRECT no mesmo filesystem dos pesos/cache, com buffer e tamanho alinhados.
import mmap, os
from pathlib import Path
probe_file = Path(__file__).parent / 'direct-io-probe.bin'
try:
    with mmap.mmap(-1, 4096) as aligned:
        fd = os.open(probe_file, os.O_CREAT | os.O_TRUNC | os.O_RDWR | os.O_DIRECT, 0o600)
        try:
            assert os.write(fd, aligned) == 4096
            os.fsync(fd)
        finally:
            os.close(fd)
finally:
    probe_file.unlink(missing_ok=True)
print('NIXL POSIX/io_uring e filesystem O_DIRECT OK', flush=True)
''')
run([PYTHON, verify], 'verify-runtime')
run([UV, 'pip', 'check', '--python', PYTHON], 'verify-dependencies')
print('Dependências verificadas. Pronto para baixar o checkpoint.')

## 3. Baixar somente os arquivos usados pelo modelo

A revisão selecionada contém **206 arquivos de pesos referenciados, totalizando 125,91 GiB**. Isso inclui os tensores BF16, a tabela PLE FP8 e MTP que fazem parte do checkpoint NVFP4; removê-los quebra o modelo. Os arquivos auxiliares são selecionados separadamente. O código recalcula os valores pela API, desconta arquivos já completos e reserva 12 GiB adicionais para JIT/NIXL e operação — uma margem operacional, não tamanho dos pesos.

O Xet faz transferências paralelas de blocos e até 32 arquivos são baixados simultaneamente. Downloads completos são reutilizados na mesma instância; arquivos incompletos são retomados pelo Hugging Face Hub.

In [ ]:
download_script = ROOT / 'download_model.py'
download_script.write_text(r'''
import os, json, shutil, hashlib
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
model_id = os.environ['PENNY_MODEL_ID']
revision = os.environ['PENNY_MODEL_REV']
local = Path(os.environ['PENNY_MODEL_DIR'])
local.mkdir(parents=True, exist_ok=True)
hf_token = os.environ.get('HF_TOKEN') or False
info = HfApi(token=hf_token).model_info(model_id, revision=revision, files_metadata=True)
assert info.sha == revision, f'Revisão inesperada: {info.sha}'
index_path = hf_hub_download(model_id, 'model.safetensors.index.json', revision=revision, local_dir=local, token=hf_token)
index = json.loads(Path(index_path).read_text())
weight_map = index.get('weight_map')
assert isinstance(weight_map, dict) and weight_map, 'weight_map inválido'
weights = set(weight_map.values())
# Não baixa relatórios, marcadores da conversão ou pesos fora do índice.
auxiliary = {
    'config.json', 'generation_config.json', 'hf_quant_config.json',
    'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json',
    'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt',
    'chat_template.jinja', 'preprocessor_config.json', 'processor_config.json',
    'video_preprocessor_config.json', 'README.md', 'LICENSE', 'LICENSE.txt',
}
metadata = {f.rfilename: f for f in info.siblings}
selected = weights | (auxiliary & metadata.keys())
# Código remoto, caso a revisão fixe auto_map; somente arquivos pequenos de código.
selected |= {name for name in metadata if name.endswith('.py')}
for name in selected:
    path = Path(name)
    assert not path.is_absolute() and '..' not in path.parts, f'Caminho inválido: {name}'
    assert name in metadata and metadata[name].size is not None, f'Metadata ausente: {name}'
weight_bytes = sum(metadata[name].size for name in weights)
total_bytes = sum(metadata[name].size for name in selected)
complete_bytes = sum(metadata[name].size for name in selected
                     if (local / name).is_file() and (local / name).stat().st_size == metadata[name].size)
remaining = total_bytes - complete_bytes
reserve = 12 * 2**30
free = shutil.disk_usage(local).free
print(f'Pesos: {len(weights)} arquivos / {weight_bytes/2**30:.2f} GiB', flush=True)
print(f'Selecionados: {total_bytes/2**30:.2f} GiB; já completos: {complete_bytes/2**30:.2f} GiB', flush=True)
print(f'Falta baixar: {remaining/2**30:.2f} GiB; margem: 12 GiB; livre: {free/2**30:.2f} GiB', flush=True)
if free < remaining + reserve:
    raise RuntimeError(f'Faltam {(remaining+reserve-free)/2**30:.2f} GiB para este download e sua margem. Escolha uma instância com mais disco.')
print('Iniciando Hugging Face/Xet em paralelo...', flush=True)
snapshot_download(model_id, revision=revision, local_dir=local, token=hf_token,
                  allow_patterns=sorted(selected), max_workers=int(os.environ['PENNY_DOWNLOAD_WORKERS']))
for name in selected:
    path = local / name
    assert path.is_file() and path.stat().st_size == metadata[name].size, f'Arquivo ausente/incompleto: {name}'
expected = '0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3'
assert hashlib.sha256((local / 'tokenizer.json').read_bytes()).hexdigest() == expected, 'Tokenizer incompatível com FR-Spec'
assert any('mtp' in k.lower() for k in weight_map), 'MTP não encontrado no índice'
manifest = {'model': model_id, 'revision': revision, 'weights_bytes': weight_bytes,
            'selected_bytes': total_bytes, 'files': sorted(selected)}
(local.parent / 'download-manifest.json').write_text(json.dumps(manifest, indent=2))
print('Checkpoint completo; tokenizer e presença do MTP verificados.', flush=True)
''')
ENV.update({'PENNY_MODEL_ID': MODEL_ID, 'PENNY_MODEL_REV': MODEL_REV,
            'PENNY_MODEL_DIR': str(MODEL), 'PENNY_DOWNLOAD_WORKERS': str(DOWNLOAD_WORKERS)})
run([PYTHON, download_script], 'download-model', timeout=14400)

## 4. Iniciar o launcher FR-Spec e testar a API local

O launcher oficial permanece intacto. O paralelismo agressivo é passado pelo ambiente; o Ninja do FlashInfer usa `MAX_JOBS=48`. O autotune permanece ativo e seus caches são preservados. Alterações nas flags do compilador podem invalidar parte dos kernels em cache e provocar recompilação na primeira execução desse perfil. Um adaptador de executável acrescenta endereço local, autenticação e configuração do loader com 24 workers. Contexto, especulação, template, parâmetros de memória e opções NIXL continuam os da receita.

A chave é renovada quando esta célula reinicia o servidor. O hash integral do checkpoint pode demorar e não imprime progresso por arquivo; há um heartbeat de espera. A célula aguarda a API responder e realiza uma geração local antes de abrir o túnel.

In [ ]:
import urllib.request, urllib.error

def stop_process(proc):
    if proc is None or proc.poll() is not None:
        return
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()

# Reexecução na mesma sessão: interrompe somente os processos deste notebook.
for process_name in ('tunnel_process', 'server_process'):
    stop_process(globals().get(process_name))
API_KEY = secrets.token_urlsafe(32)
ENV['PENNY_API_KEY'] = API_KEY
ENV['PENNY_REAL_SGLANG'] = str(Path(PYBIN) / 'sglang')
ENV['PENNY_WEIGHT_LOADER_CONFIG'] = json.dumps({
    'enable_multithread_load': True, 'num_threads': WEIGHT_LOAD_WORKERS,
})
print(f'G4: build jobs={BUILD_JOBS}; NVCC threads={NVCC_THREADS}; loader workers={WEIGHT_LOAD_WORKERS}', flush=True)
if not Path(ENV['PENNY_REAL_SGLANG']).is_file():
    raise RuntimeError('Executável sglang ausente no Python do servidor.')
wrapper = ROOT / 'sglang-auth'
wrapper.write_text('#!/usr/bin/env bash\nset -euo pipefail\n'
                   'exec "$PENNY_REAL_SGLANG" "$@" --host 127.0.0.1 --api-key "$PENNY_API_KEY" --model-loader-extra-config "$PENNY_WEIGHT_LOADER_CONFIG"\n')
wrapper.chmod(0o700)
ENV.update({
    'REPO_ROOT': str(REPO), 'PYTHON': PYTHON, 'SGLANG_EXE': str(wrapper),
    'TARGET_MODEL': str(MODEL), 'CACHE_BASE': str(ROOT / 'cache'),
    'NIXL_STORAGE_BASE': str(ROOT / 'nixl-cache'),
    'PENNY_PLE_BACKEND': 'ram', 'SGLANG_MM_PREPROCESS_DEVICE': 'cpu',
    'SGLANG_SM120_ONLINE_MXFP8': str(ONLINE_FP8).lower(),
})
launcher = REPO / 'configs/pennyroyal/serve-flash-next-frspec.sh'
run(['bash', '-n', launcher], 'check-launcher')
server_log = LOGS / 'server.log'
with server_log.open('w') as output:
    server_process = subprocess.Popen(['bash', str(launcher)], env=ENV, cwd=REPO,
                                      stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
server_log.chmod(0o600)
LOCAL_URL = 'http://127.0.0.1:8001'

def api_request(base, path, *, payload=None, key=API_KEY, timeout=30):
    headers = {}
    if key is not None:
        headers['Authorization'] = 'Bearer ' + key
    data = None
    if payload is not None:
        data = json.dumps(payload).encode()
        headers['Content-Type'] = 'application/json'
    request = urllib.request.Request(base + path, data=data, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.status, json.loads(response.read())
    except urllib.error.HTTPError as error:
        return error.code, error.read().decode(errors='replace')

started = time.monotonic()
last_notice = started
try:
    with server_log.open() as reader:
        while True:
            text = reader.read()
            if text:
                print(text.replace(API_KEY, '[CHAVE OCULTA]'), end='', flush=True)
            if server_process.poll() is not None:
                raise RuntimeError(f'Servidor encerrou com código {server_process.returncode}; log: {server_log}')
            try:
                status, body = api_request(LOCAL_URL, '/v1/models', timeout=3)
                if status == 200 and any(m['id'] == 'pennyroyal' for m in body.get('data', [])):
                    break
            except (OSError, ValueError):
                pass
            elapsed = time.monotonic() - started
            if elapsed > STARTUP_TIMEOUT:
                raise TimeoutError(f'Servidor não ficou pronto em {STARTUP_TIMEOUT}s; consulte {server_log}')
            if time.monotonic() - last_notice >= 30:
                print(f'Inicializando: {elapsed/60:.1f} min; hash/JIT/CUDA graphs podem demorar.', flush=True)
                last_notice = time.monotonic()
            time.sleep(2)
    for invalid_key in (None, 'invalid-test-key'):
        status, body = api_request(LOCAL_URL, '/v1/models', key=invalid_key)
        assert status in (401, 403), f'Falha de autenticação: chave inválida recebeu HTTP {status}'
    TEST_PAYLOAD = {
        'model': 'pennyroyal',
        'messages': [{'role': 'user', 'content': 'Responda apenas com a palavra PRONTO.'}],
        'temperature': 0, 'max_tokens': 32, 'stream': False,
        'chat_template_kwargs': {'enable_thinking': False},
    }
    status, body = api_request(LOCAL_URL, '/v1/chat/completions', payload=TEST_PAYLOAD, timeout=600)
    assert status == 200, f'Geração local falhou: HTTP {status}: {body}'
    assert body.get('choices') and body['choices'][0].get('message', {}).get('content'), body
    print('\nGeração local OK:', body['choices'][0]['message']['content'])
except BaseException:
    stop_process(server_process)
    raise

## 5. Publicar com Cloudflare, validar e mostrar os dados de conexão

O Quick Tunnel cria uma URL HTTPS aleatória, sem cadastro. A célula só mostra **API PÚBLICA VALIDADA** após três verificações: `/v1/models` autenticado, rejeição de chave ausente/incorreta e uma geração de texto pela Internet.

As credenciais aparecerão na saída para você copiar. Ao compartilhar o notebook depois de executá-lo, remova as saídas. A chave não tem prazo em horas: sua validade termina ao parar/reiniciar este servidor. O Quick Tunnel deixa de encaminhar quando seu processo termina.

In [ ]:
if server_process.poll() is not None:
    raise RuntimeError('Servidor não está ativo. Execute a célula anterior.')
stop_process(globals().get('tunnel_process'))
cloudflared = ROOT / 'cloudflared'
# Resolve uma versão oficial e confere o SHA-256 publicado nos metadados do release.
release_request = urllib.request.Request(
    'https://api.github.com/repos/cloudflare/cloudflared/releases/latest',
    headers={'Accept': 'application/vnd.github+json', 'User-Agent': 'Pennyroyal-Colab'})
with urllib.request.urlopen(release_request, timeout=60) as response:
    release = json.load(response)
asset = next(a for a in release['assets'] if a['name'] == 'cloudflared-linux-amd64')
run(['curl', '--fail', '--location', '--retry', '5', '--output', cloudflared,
     asset['browser_download_url']], 'download-cloudflared')
import hashlib
actual_hash = hashlib.sha256(cloudflared.read_bytes()).hexdigest()
digest = asset.get('digest')
if digest and digest.startswith('sha256:'):
    assert actual_hash == digest.removeprefix('sha256:'), 'Checksum cloudflared incorreto'
else:
    raise RuntimeError('Release não publicou digest SHA-256 do binário; download não aprovado pelo verificador.')
cloudflared.chmod(0o700)
print('cloudflared', release['tag_name'], 'SHA-256 verificado')
# Config explícita vazia evita interferência de config.yml/config.yaml preexistente.
empty_config = ROOT / 'cloudflared-empty.yml'
empty_config.write_text('{}\n')
tunnel_log = LOGS / 'cloudflared.log'
with tunnel_log.open('w') as output:
    tunnel_process = subprocess.Popen([
        str(cloudflared), 'tunnel', '--config', str(empty_config), '--no-autoupdate',
        '--protocol', 'http2', '--url', LOCAL_URL,
    ], stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
started = time.monotonic()
PUBLIC_URL = None
try:
    while time.monotonic() - started < 180:
        if tunnel_process.poll() is not None:
            raise RuntimeError('Cloudflare encerrou:\n' + tunnel_log.read_text())
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log.read_text())
        if match:
            PUBLIC_URL = match.group(0)
            break
        time.sleep(2)
    if not PUBLIC_URL:
        raise TimeoutError('Cloudflare não gerou URL em 180s:\n' + tunnel_log.read_text())
    last_error = None
    for attempt in range(60):
        if tunnel_process.poll() is not None or server_process.poll() is not None:
            raise RuntimeError('Servidor ou túnel encerrou durante a verificação pública.')
        try:
            status, body = api_request(PUBLIC_URL, '/v1/models', timeout=10)
            if status == 200 and any(m['id'] == 'pennyroyal' for m in body.get('data', [])):
                break
            last_error = (status, body)
        except (OSError, ValueError) as error:
            last_error = str(error)
        if attempt % 5 == 0:
            print(f'Aguardando propagação da URL temporária, tentativa {attempt+1}/60...', flush=True)
        time.sleep(2)
    else:
        raise RuntimeError(f'API pública não respondeu corretamente: {last_error}')
    for invalid_key in (None, 'invalid-test-key'):
        status, body = api_request(PUBLIC_URL, '/v1/models', key=invalid_key)
        assert status in (401, 403), f'Autenticação pública inválida: HTTP {status}'
    status, body = api_request(PUBLIC_URL, '/v1/chat/completions', payload=TEST_PAYLOAD, timeout=120)
    assert status == 200, f'Geração pública falhou: HTTP {status}: {body}'
    assert body.get('choices') and body['choices'][0].get('message', {}).get('content'), body
except BaseException:
    stop_process(tunnel_process)
    raise

print('\nAPI PÚBLICA VALIDADA')
print('Base URL:', PUBLIC_URL + '/v1')
print('API key:', API_KEY)
print('Model: pennyroyal')
print('Resposta:', body['choices'][0]['message']['content'])
print('\nCopie para bash/zsh (Arch Linux/macOS):\n')
print('export OPENAI_BASE_URL=' + shlex.quote(PUBLIC_URL + '/v1'))
print('export OPENAI_API_KEY=' + shlex.quote(API_KEY))
print(r'''
curl --fail-with-body --max-time 120 "$OPENAI_BASE_URL/chat/completions" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -H 'Content-Type: application/json' \
  -d '{"model":"pennyroyal","messages":[{"role":"user","content":"Responda apenas PRONTO."}],"max_tokens":32,"stream":false,"chat_template_kwargs":{"enable_thinking":false}}'
''')
print('A instância deve permanecer ativa. Encerrar a sessão Colab remove o serviço.')

## 6. Controles da sessão

Executar esta célula apenas mostra os botões. **Executar tudo não encerra a API.** Use “Encerrar API” quando terminar. “Ver logs” mostra as últimas linhas e o estado dos processos.

**Cliente Python, Linux/macOS/Windows:** depois de copiar a URL e a chave, use a biblioteca padrão ou `openai`. O nome exposto é `pennyroyal`, conforme o launcher — não o identificador Hugging Face. Agentes que exigem streaming SSE precisam de um túnel Cloudflare nomeado, que exige configuração de conta/domínio; isso não é fornecido pelo Quick Tunnel.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
controls_output = widgets.Output()
stop_button = widgets.Button(description='Encerrar API', button_style='danger')
logs_button = widgets.Button(description='Ver logs')

def shutdown(_):
    with controls_output:
        stop_process(globals().get('tunnel_process'))
        stop_process(globals().get('server_process'))
        ENV.pop('PENNY_API_KEY', None)
        print('Túnel e servidor encerrados. A chave anterior não dá acesso a uma API ativa.')

def show_logs(_):
    with controls_output:
        for name in ('server_process', 'tunnel_process'):
            proc = globals().get(name)
            print(name, 'ativo' if proc is not None and proc.poll() is None else 'encerrado')
        for filename in ('server.log', 'cloudflared.log'):
            print('\n' + filename)
            path = LOGS / filename
            if path.exists():
                print('\n'.join(path.read_text(errors='replace').splitlines()[-35:]).replace(API_KEY, '[CHAVE OCULTA]'))
stop_button.on_click(shutdown)
logs_button.on_click(show_logs)
display(widgets.HBox([logs_button, stop_button]), controls_output)

## Fontes e diferenças deliberadas

1. [Receita e requisitos oficiais](https://github.com/jpezzulli/sglang-rtxpro6000/blob/pennyroyal-v2.5.0/BUILD.md): versões, instalação nativa, revisão do checkpoint e NIXL POSIX.
2. [Launcher FR-Spec utilizado](https://github.com/jpezzulli/sglang-rtxpro6000/blob/pennyroyal-v2.5.0/configs/pennyroyal/serve-flash-next-frspec.sh): parâmetros de inferência, checksums e namespace NIXL.
3. [Checkpoint RadixArk](https://huggingface.co/RadixArk/Qwen3.8-Flash-Next-NVFP4/tree/7b719225242aacd3dbd3f9407468c2ee9a9d2594): pesos, índice e tokenizer.
4. [Cloudflare Quick Tunnels](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/do-more-with-tunnels/trycloudflare/): URL temporária, limite de requisições e ausência de suporte SSE.
5. [Compatibilidade CUDA](https://docs.nvidia.com/cuda/cuda-toolkit-release-notes/index.html): toolkit e driver. A [matriz de forward compatibility](https://docs.nvidia.com/deploy/cuda-compatibility/forward-compatibility.html) lista `cuda-compat-13-3` como compatível com R580. O notebook instala esse pacote quando necessário e testa o carregamento da biblioteca e PTX JIT; não confunde compatibilidade parcial de runtime com suporte a PTX novo.

**Adaptações:** build sem isolamento limitado ao pacote SGLang, mantendo isolamento e instalação dos backends declarados das dependências; Ubuntu no lugar de Fedora; Python separado sem venv; instalação automática de toolchain; testes e exemplos; perfil de compilação G4 de 48 jobs e NVCC com quatro threads; loader com 24 workers; autenticação e bind local adicionados por wrapper; Cloudflare Quick Tunnel. NIXL é compilado contra o Torch já instalado, sem o isolamento upstream que pede Torch 2.11, e dispensa seus exemplos/testes internos; verifica importação, inicialização do plugin POSIX/io_uring e escrita O_DIRECT. A inicialização real exercita o caminho do servidor. A validação final não é benchmark nem validação de 524K, vision ou restauração de cache.

**Persistência:** HiCache/NIXL e caches JIT ficam no disco local da instância. Reiniciar só o servidor conserva esses arquivos; descartar a VM perde pesos e caches. Nenhuma montagem do Google Drive é necessária. Não há mecanismo de contorno de limites de sessão Colab.

**Reprodutibilidade:** fonte SGLang, revisão dos pesos e fonte NIXL fixadas. Dependências seguem os pins do projeto, mas o autor não fornece um lock completo; pacotes transitivos e binário Cloudflare podem mudar. `uv pip check`, testes CUDA/PTX, autenticação e geração são barreiras de validação, não uma garantia de execução em qualquer imagem futura do Colab.